# Phase W1 — CovariateConditionedRanders on Sim2Real-Fire (Flat Grid)

**Scientific question:** Can a Lagrangian covariate-conditioned Randers metric, trained via differentiable geodesic AVBD solvers, predict wildfire arrival times with Pearson r ≥ 0.70, and does the Finsler wind drift term provide a statistically significant improvement over the isotropic Riemannian baseline?

**Reference:** Gahtan, Shpund & Bronstein (2026). *Wildfire Simulation with Differentiable Randers-Finsler Eikonal Solvers.* arXiv:2603.00035.

---

## Architecture at a glance

```
CovariateConditionedRanders(x, v) = √[gᵢⱼ(x)vⁱvʲ] + bᵢ(x)vⁱ
```
where `g` (PSD matrix field) and `b` (drift/wind vector field) are outputs of a CNN
that takes terrain rasters (elevation, slope, aspect, canopy, fuel) and a weather vector.
The geodesic arc length from ignition to each burned pixel is regressed against the
observed pixel arrival time via the **AVBD** boundary-value solver with implicit differentiation.

---

## Runtime guide

| Mode | Data needed | ~Time (T4 GPU) |
|------|------------|----------------|
| `QUICK = True` + `USE_SYNTHETIC = True` | none | ~5 min |
| `QUICK = False` + `USE_SYNTHETIC = True` | none | ~30 min |
| `QUICK = False` + `USE_SYNTHETIC = False` | Sim2Real-Fire dataset | 2–8 h per scene |

In [1]:
# ============================================================
# CELL 1 — Colab environment setup
# Run this first. Runtime → Change runtime type → GPU (T4 or A100)
# ============================================================

import subprocess, sys

def _run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result.returncode == 0

# --- Install JAX with CUDA support ---
print("Installing JAX (CUDA 12)...")
_run("pip install -q --upgrade 'jax[cuda12]'")

# --- Install HAMTools ---
# Option A (recommended): install directly from GitHub
print("Installing HAMTools...")
_run("pip install -q git+https://github.com/hubibala/ham.git")

# Option B: if you cloned the repo to Google Drive, uncomment and use instead:
# import sys; sys.path.insert(0, '/content/drive/MyDrive/HAM/src')

# --- Wildfire optional dependencies ---
print("Installing rasterio + Pillow (wildfire data loaders)...")
_run("pip install -q 'Pillow>=9.0' 'rasterio>=1.3'")

print("Done. Restart runtime if JAX was upgraded (Runtime → Restart session).")

Installing JAX (CUDA 12)...
Installing HAMTools...
Installing rasterio + Pillow (wildfire data loaders)...
Done. Restart runtime if JAX was upgraded (Runtime → Restart session).


In [3]:
# ============================================================
# CELL 2 — Verify GPU is available
# ============================================================
import jax

# configure_device was added in a recent HAMTools commit.
# If the installed version is older, define it inline here.
try:
    from ham.utils import configure_device
except ImportError:
    def configure_device(device: str):
        dev = jax.devices(device)[0]
        jax.config.update("jax_default_device", dev)
        print(f"[HAMTools inline] Default JAX device set to: {dev}")
        return dev

devices = jax.devices()
print(f"Available JAX devices: {devices}")

# Configure device — change to 'cpu' if no GPU is attached
DEVICE = "gpu" if any("cuda" in str(d).lower() for d in devices) else "cpu"
configure_device(DEVICE)
print(f"Using device: {DEVICE}")

Available JAX devices: [CudaDevice(id=0)]
[HAMTools] Default JAX device set to: cuda:0
Using device: gpu


In [4]:
# ============================================================
# CELL 3 — Experiment configuration
# ============================================================

# ---- Main switches ----
QUICK         = True   # True = smoke test (~5 min); False = full experiment
USE_SYNTHETIC = True   # True = no dataset needed; False = real Sim2Real-Fire data
USE_WIND      = True   # True = Randers (wind drift); False = Riemannian ablation
OUTPUT_DIR    = "results/phaseW1"

# Real-data settings (only used when USE_SYNTHETIC = False)
# Option A: mount Google Drive
#   from google.colab import drive; drive.mount('/content/drive')
#   DATA_ROOT = '/content/drive/MyDrive/sim2real_fire'
# Option B: download from Gahtan et al. repo
#   !git clone --depth=1 https://github.com/BarakGahtan/differentiable-eikonal-wildfire /content/gahtan
#   DATA_ROOT = '/content/gahtan/experiments/wildfire/data'
DATA_ROOT  = "data/sim2real_fire"
SCENE_IDS  = ["0001"]   # list of scene folder names inside DATA_ROOT

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/figs", exist_ok=True)
print(f"Config: QUICK={QUICK}, USE_SYNTHETIC={USE_SYNTHETIC}, USE_WIND={USE_WIND}")

Config: QUICK=True, USE_SYNTHETIC=True, USE_WIND=True


In [12]:
# ============================================================
# CELL 4 — Imports
# ============================================================
import sys, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

import jax
import jax.numpy as jnp
import equinox as eqx
import optax
from jax import config
config.update("jax_enable_x64", True)

# HAMTools core
from ham.geometry.manifolds import EuclideanSpace
from ham.solvers.avbd import AVBDSolver
from ham.training.losses import ArrivalTimeLoss

ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if ON_COLAB:
    _examples_path = "/content/HAM"

else:
# Running from examples/notebooks/ locally
    _examples_path = os.path.abspath(os.path.join(os.path.dirname("file"), ".."))

if _examples_path not in sys.path:
    sys.path.insert(0, _examples_path)

print(f"Looking for experiment scripts in: {_examples_path}")

try:
    from experiment_wildfire_flat import (
        get_config, make_metric, make_solver,
        bind_scenario_to_metric,
        _make_synthetic_scenario,
        _pixels_to_world, _ignition_to_world,
        pearson_r,
        _predict_arrivals_chunked,
        run_synthetic, run_experiment,
    )
    print("Imported experiment_wildfire_flat OK")
except ImportError as e:
    print(f"Import error: {e}")
    print(f"Contents of {_examples_path}:")
    print([f for f in os.listdir(_examples_path) if f.endswith(".py")])

Looking for experiment scripts in: /content/HAM
Import error: No module named 'experiment_wildfire_flat'
Contents of /content/HAM:


FileNotFoundError: [Errno 2] No such file or directory: '/content/HAM'

In [ ]:
# ============================================================
# CELL 5 — Build configuration
# ============================================================
cfg = get_config(quick=QUICK)

# Adjust batch size for GPU memory
if DEVICE == "gpu":
    cfg["batch_size_fires"] = 32 if not QUICK else 16
    cfg["k_train_obs"] = 200 if not QUICK else 100

print("Configuration:")
for k, v in cfg.items():
    print(f"  {k:35s} = {v}")

In [ ]:
# ============================================================
# CELL 6 — Run experiment
# ============================================================
if USE_SYNTHETIC:
    print("Running synthetic smoke test (no real dataset needed)...")
    result = run_synthetic(cfg, output_dir=OUTPUT_DIR, use_wind=USE_WIND)
    all_results = [result]
else:
    print(f"Running real-data experiment on scenes: {SCENE_IDS}")
    all_results = run_experiment(
        data_root=DATA_ROOT,
        scene_ids=SCENE_IDS,
        output_dir=OUTPUT_DIR,
        cfg=cfg,
        use_wind=USE_WIND,
    )

print("\n=== Final Results ===")
for r in all_results:
    print(f"  scene={r['scene_id']}  "
          f"Pearson r={r['test_pearson_r_mean']:.4f}±{r['test_pearson_r_std']:.4f}  "
          f"IoU@50={r.get('test_iou50', float('nan')):.4f}  "
          f"wind={'yes' if r.get('use_wind') else 'no'}")

In [ ]:
# ============================================================
# CELL 7 — Training loss convergence
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for r in all_results[:3]:
    label = f"{r['scene_id']} seed={r.get('seed', '?')}"
    epochs = np.arange(1, len(r["train_loss_history"]) + 1)
    axes[0].plot(epochs, r["train_loss_history"], label=label)
    if r.get("val_r_history"):
        axes[1].plot(np.arange(1, len(r["val_r_history"]) + 1),
                     r["val_r_history"], label=label)

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train loss (MSE)")
axes[0].set_title("Loss convergence"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Pearson r")
axes[1].set_title("Validation correlation"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/w1_convergence.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# CELL 8 — Per-scene correlation bar chart
# ============================================================
scene_ids_unique = sorted({r["scene_id"] for r in all_results})
r_means = [np.mean([r["test_pearson_r_mean"] for r in all_results if r["scene_id"] == sid])
           for sid in scene_ids_unique]
r_stds  = [np.std([r["test_pearson_r_mean"] for r in all_results if r["scene_id"] == sid])
           for sid in scene_ids_unique]

fig, ax = plt.subplots(figsize=(max(6, len(scene_ids_unique) * 1.2), 4))
x = np.arange(len(scene_ids_unique))
ax.bar(x, r_means, yerr=r_stds, capsize=5, color="#4477AA", alpha=0.85, label="HAMTools W1")
ax.axhline(0.70,  color="red",    linestyle="--", linewidth=1.5, label="Target r=0.70")
ax.axhline(0.824, color="orange", linestyle=":",  linewidth=1.5, label="Gahtan et al. (0.824)")
ax.set_xticks(x); ax.set_xticklabels(scene_ids_unique, rotation=30, ha="right")
ax.set_ylabel("Test Pearson r"); ax.set_ylim(0, 1.05)
ax.set_title("Phase W1 — Arrival-time correlation per scene")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/w1_correlation_bar.png", dpi=120)
plt.show()

print("\nSummary:")
for sid, rm, rs in zip(scene_ids_unique, r_means, r_stds):
    status = "PASS" if rm >= 0.70 else "FAIL"
    print(f"  [{status}] {sid}: r = {rm:.4f} ± {rs:.4f}")

In [ ]:
# ============================================================
# CELL 9 — Arrival-time heatmap (synthetic mode only)
# ============================================================
if USE_SYNTHETIC and all_results and "pred_grid" in all_results[0]:
    r = all_results[0]
    pred_grid = np.array(r["pred_grid"])
    gt_grid   = np.array(r["gt_grid"])
    pred_norm = (pred_grid - pred_grid.min()) / (pred_grid.max() - pred_grid.min() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    im0 = axes[0].imshow(gt_grid, cmap="hot_r", vmin=0, vmax=1)
    axes[0].set_title("Ground truth arrival time")
    plt.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(pred_norm, cmap="hot_r", vmin=0, vmax=1)
    axes[1].set_title(f"Predicted (r={r['test_pearson_r_mean']:.3f})")
    plt.colorbar(im1, ax=axes[1])

    err = np.abs(pred_norm - gt_grid)
    im2 = axes[2].imshow(err, cmap="Reds")
    axes[2].set_title("Absolute error")
    plt.colorbar(im2, ax=axes[2])

    plt.suptitle("Phase W1 — Synthetic smoke test heatmap", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/figs/w1_heatmap.png", dpi=120)
    plt.show()
else:
    print("Heatmap requires USE_SYNTHETIC=True and 'pred_grid' key in results.")
    print("For real-data runs, per-scene figures are saved to OUTPUT_DIR/figs/ automatically.")

In [ ]:
# ============================================================
# CELL 10 — Save checkpoint
# ============================================================
import equinox as eqx

CHECKPOINT_PATH = f"{OUTPUT_DIR}/metric_w1_seed{cfg['seed']}.eqx"

if "metric" in dir() and metric is not None:
    eqx.tree_serialise_leaves(CHECKPOINT_PATH, metric)
    print(f"Checkpoint saved: {CHECKPOINT_PATH}")
    print("Load this in the W3 geometric analysis notebook with:")
    print(f"  metric = eqx.tree_deserialise_leaves('{CHECKPOINT_PATH}', metric_template)")
else:
    print("No 'metric' variable found in scope.")
    print("The run_synthetic / run_experiment functions save checkpoints automatically to OUTPUT_DIR.")

# To copy to Google Drive on Colab:
# import shutil
# shutil.copy(CHECKPOINT_PATH, '/content/drive/MyDrive/ham_checkpoints/')